# RQ-VAE longitudinal checkpoints and GPT2Rec plan

This notebook is a new standalone experiment notebook. It does not replace the original RQVAE_EarlyStopping_Claude notebook.

Design: train 3 independent RQ-VAE seeds, save epoch 1, 3, 5, 10, best, and final checkpoints, then create a downstream GPT2Rec sweep plan with tie_break=count and GPT2 seeds 0, 1, 2.

Important epoch rule: best and final are different concepts. best is the lowest validation-loss state. final is the last actually trained state before any best-state restore.


In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils import data
from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import random


In [ ]:
KAGGLE_DATA = '/kaggle/input/datasets/qwerte123/hetero-data-updated-diffemb/heterodata_object12_updated.pt'
LOCAL_DATA  = r'C:\Users\Admin\Documents\GitHub\PLUM_implementation\data\heterodata_object12_updated.pt'
DATA_PATH   = KAGGLE_DATA if os.path.exists(KAGGLE_DATA) else LOCAL_DATA

df = torch.load(DATA_PATH, weights_only=False, map_location='cpu')
df

In [ ]:
class Encoder(nn.Module):
    """MLP encoder/decoder with LayerNorm after each hidden layer."""
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers = []
        dims = [input_dim] + list(hidden_dims) + [output_dim]
        for i, (in_d, out_d) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(in_d, out_d, bias=True))
            if i < len(dims) - 2:          # not last layer
                layers.append(nn.LayerNorm(out_d))
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
class EMACodebook(nn.Module):
    """
    Vector-quantization codebook with Exponential Moving Average updates.
    - Codebook entries tracked as buffers (not optimizer parameters)
    - K-means initialization on the first training batch
    - Laplace smoothing to keep dead codes alive
    - Cosine distance for assignment (scale-invariant)
    """
    def __init__(self, codebook_size, emb_dim, beta=0.25, ema_decay=0.99, epsilon=1e-4):
        super().__init__()
        self.codebook_size = codebook_size
        self.beta          = beta
        self.ema_decay     = ema_decay
        self.epsilon       = epsilon

        emb = F.normalize(torch.randn(codebook_size, emb_dim), p=2, dim=1)
        self.register_buffer('emb',         emb)
        self.register_buffer('ema_count',   torch.ones(codebook_size))
        self.register_buffer('ema_weight',  emb.clone())
        self.register_buffer('initialized', torch.zeros(1, dtype=torch.bool))

    @torch.no_grad()
    def _kmeans_init(self, x):
        """Initialize codebook with a random sample of encoder outputs."""
        n = x.shape[0]
        if n >= self.codebook_size:
            idx = torch.randperm(n, device=x.device)[:self.codebook_size]
            init = x[idx]
        else:
            repeats = (self.codebook_size // n) + 1
            init    = x.repeat(repeats, 1)[:self.codebook_size]
        self.emb.copy_(F.normalize(init, p=2, dim=1))
        self.ema_weight.copy_(self.emb)
        self.initialized.fill_(True)

    @torch.no_grad()
    def _ema_update(self, x, ids):
        one_hot = F.one_hot(ids, self.codebook_size).float()  
        n_k = one_hot.sum(0)                               
        m_k = one_hot.T @ x                               

        self.ema_count.mul_(self.ema_decay).add_(n_k   * (1 - self.ema_decay))
        self.ema_weight.mul_(self.ema_decay).add_(m_k  * (1 - self.ema_decay))

        n_total = self.ema_count.sum()
        count_smooth = (
            (self.ema_count + self.epsilon) /
            (n_total + self.codebook_size * self.epsilon) * n_total
        )
        self.emb.copy_(self.ema_weight / count_smooth.unsqueeze(1))

    def forward(self, x):
        if self.training and not self.initialized:
            self._kmeans_init(x.detach())

        x_n = F.normalize(x,        p=2, dim=1)
        code_n = F.normalize(self.emb, p=2, dim=1)
        dist = 1.0 - x_n @ code_n.T          
        ids = dist.argmin(dim=1)           
        emb = self.emb[ids]                 

        if self.training:
            self._ema_update(x.detach(), ids)

        commitment_loss = self.beta * F.mse_loss(x, emb.detach())
        emb_st = x + (emb - x).detach()        
        return commitment_loss, emb_st, ids

In [ ]:
class RQVAE_Improved(nn.Module):
    def __init__(self, inp_size, hidden_sizes, embed_dim, n_layers,
                 codebook_size=256, beta=0.25, gamma=0.1,
                 ema_decay=0.99, temperature=0.07):
        super().__init__()
        self.n_layers    = n_layers
        self.gamma       = gamma
        self.temperature = temperature

        self.enc = Encoder(inp_size,   hidden_sizes,           embed_dim)
        self.dec = Encoder(embed_dim,  hidden_sizes[::-1],     inp_size)
        self.codebooks = nn.ModuleList([
            EMACodebook(codebook_size, embed_dim, beta=beta, ema_decay=ema_decay)
            for _ in range(n_layers)
        ])

    def forward(self, x, related, is_self_pair=None):
        x_n = F.normalize(x,       p=2, dim=1)
        r_n = F.normalize(related, p=2, dim=1)

        encoded = self.enc(x_n)
        rel_encoded = self.enc(r_n)

        if self.training:
            level_mask = torch.randint(1, self.n_layers + 1, (1,)).item()
        else:
            level_mask = self.n_layers

        r = encoded
        e_sum = torch.zeros_like(encoded)
        q_losses = x.new_zeros([])
        sids = []

        for level, cb in enumerate(self.codebooks):
            q_loss, emb_st, ids = cb(r)
            r = r - emb_st.detach()     
            sids.append(ids)
            if level < level_mask:      
                q_losses = q_losses + q_loss
                e_sum    = e_sum    + emb_st

        dec = self.dec(e_sum)
        rec_loss = F.mse_loss(x_n, dec)

        enc_n = F.normalize(encoded, p=2, dim=1)
        rel_n = F.normalize(rel_encoded, p=2, dim=1)
        sims = (enc_n @ rel_n.T) /self.temperature   
        log_p = F.log_softmax(sims, dim=1)
        diag  = -torch.diag(log_p)                     

        if is_self_pair is not None:
            valid = ~is_self_pair
            con_loss = diag[valid].mean() if valid.any() else x.new_zeros([])
        else:
            con_loss = diag.mean()

        loss = q_losses + rec_loss + self.gamma * con_loss

        return {
            'loss': loss,
            'q_loss': q_losses,
            'r_loss': rec_loss,
            'con_loss': con_loss,
            'sids': sids,
            'level_mask': level_mask,
        }

In [ ]:
class Data_Embeddings(Dataset):
    def __init__(self, embed, related):
        self.embed   = embed
        self.related = related

    def __len__(self):
        return self.embed.shape[0]

    def __getitem__(self, index):
        rel_ids = self.related[index]
        if len(rel_ids) == 0:
            rel_embed = self.embed[index]
            is_self   = True
        else:
            rel_embed = self.embed[int(np.random.choice(rel_ids))]
            is_self   = False
        return rel_embed, self.embed[index], torch.tensor(is_self)

In [ ]:
def unique_per_level(sids):
    return torch.tensor([s.unique().numel() for s in sids])

def entropy_per_level(all_sids, codebook_size):
    out = []
    for level_ids in all_sids:
        ids = torch.cat(level_ids).view(-1)
        counts = torch.bincount(ids, minlength=codebook_size).float()
        probs = counts / counts.sum()
        probs = probs[probs > 0]
        out.append(-(probs * probs.log2()).sum())
    return torch.stack(out).cpu()

def sids_analyze(model, ds, device):
    model.eval()
    sid_to_ids = defaultdict(list)
    id_to_sid  = {}
    total_loss = 0.0
    with torch.no_grad():
        for item_id in range(len(ds)):
            rel_emb, emb, is_self = ds[item_id]
            emb = emb.to(device).unsqueeze(0)
            rel_emb = rel_emb.to(device).unsqueeze(0)
            out = model(emb, rel_emb, is_self.unsqueeze(0))
            sid = tuple(s.item() for s in out['sids'])
            sid_to_ids[sid].append(item_id)
            id_to_sid[item_id] = sid
            total_loss += out['loss'].item()

    id_to_fullsid = {}
    for sid, ids in sid_to_ids.items():
        for cnt, iid in enumerate(ids):
            id_to_fullsid[iid] = sid if len(ids) == 1 else (*sid, cnt)

    return {'id_to_sid': id_to_sid, 'sid_to_ids': sid_to_ids,
            'id_to_fullsid': id_to_fullsid,
            'total_loss': total_loss / len(ds)}

def calculate_pas(sid_to_ids, embeddings):
    """Mean pairwise cosine similarity within colliding SID clusters (PAS)."""
    sims = []
    emb = F.normalize(embeddings.float(), p=2, dim=1)
    for ids in sid_to_ids.values():
        if len(ids) < 2:
            continue
        vecs = emb[ids]
        sim_matrix = vecs @ vecs.T
        k = len(ids)
        idx = torch.triu_indices(k, k, offset=1)
        sims.append(sim_matrix[idx[0], idx[1]])
    if not sims:
        return float('nan')
    return torch.cat(sims).mean().item()

def calculate_metrics(sid_to_ids, n_items, embeddings=None,
                      codebook_size=None, n_layers=None):
    n_colls = sum(len(v) - 1 for v in sid_to_ids.values())
    metrics = {
        'n_unique_sid':         len(sid_to_ids),
        'n_collisions':         n_colls,
        'max_cluster':          max(len(v) for v in sid_to_ids.values()),
        'collision_free_ratio': (n_items - n_colls) / n_items,
        'icr':                  len(sid_to_ids) / n_items,
    }
    if codebook_size is not None and n_layers is not None:
        total_paths = codebook_size ** n_layers
        metrics['sid_coverage'] = len(sid_to_ids) / total_paths
    if embeddings is not None:
        metrics['pas'] = calculate_pas(sid_to_ids, embeddings)
    return metrics

In [ ]:
embeds = df['item'].x
related = df['item'].related
ds = Data_Embeddings(embeds, related)

train_ds, val_ds = data.random_split(
    ds, [0.9, 0.1], generator=torch.Generator().manual_seed(42)
)
dl_train = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=0)
dl_val = DataLoader(val_ds,   batch_size=len(val_ds), shuffle=False, num_workers=0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}  items={len(ds)}  train={len(train_ds)}  val={len(val_ds)}')

In [ ]:
# Longitudinal RQ-VAE experiment config.
# Main design:
#   RQ-VAE seeds: 3 independent runs
#   checkpoints: epoch 1, 3, 5, 10, best, final
#   main downstream tie-break: count
#   downstream GPT2Rec seeds: 0, 1, 2
#
# Epoch logic:
#   milestone checkpoints are saved only after the exact epoch has been trained.
#   best is the lowest validation-loss state across the whole run.
#   final is the last actually trained epoch before any best-state restore.
#   with patience=20, epoch 10 is practically guaranteed unless n_ep is reduced.

COURSEWORK_ROOT = Path(r'C:\Users\Admin\Documents\GitHub\plum_sid_coursework')
RQVAE_CKPT_DIR = COURSEWORK_ROOT / 'checkpoints' / 'rqvae_longitudinal_l4'
RQVAE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RQVAE_CKPT_DIR / 'rqvae_longitudinal_l4_manifest.csv'
GPT2_PLAN_PATH = RQVAE_CKPT_DIR / 'gpt2_count_sweep_plan.csv'

n_layers = 4
codebook_size = 256
embed_dim = 32
hidden_sizes = [1024, 512, 256, 128]
beta = 0.25
gamma = 0.1
ema_decay = 0.99
temperature = 0.07
lr = 1e-3

rqvae_seeds = [0, 1, 2]
gpt2_seeds = [0, 1, 2]
main_tie_break = 'count'
checkpoint_epochs = [1, 3, 5, 10]
n_ep = 200
patience = 20
min_delta = 1e-4
batch_size = 512

assert max(checkpoint_epochs) <= n_ep, 'n_ep must cover the largest milestone epoch'
print('RQVAE_CKPT_DIR:', RQVAE_CKPT_DIR)
print('RQ-VAE seeds:', rqvae_seeds)
print('checkpoint epochs:', checkpoint_epochs)
print('main tie_break for GPT2Rec:', main_tie_break)
print('GPT2Rec seeds:', gpt2_seeds)


In [ ]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def make_loaders(seed):
    gen = torch.Generator().manual_seed(seed)
    dl_train_seeded = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=gen,
    )
    dl_val_seeded = DataLoader(
        val_ds,
        batch_size=len(val_ds),
        shuffle=False,
        num_workers=0,
    )
    return dl_train_seeded, dl_val_seeded


def make_model(seed):
    set_global_seed(seed)
    return RQVAE_Improved(
        inp_size=embeds.shape[1],
        hidden_sizes=hidden_sizes,
        embed_dim=embed_dim,
        n_layers=n_layers,
        codebook_size=codebook_size,
        beta=beta,
        gamma=gamma,
        ema_decay=ema_decay,
        temperature=temperature,
    ).to(device)


def run_epoch(model, loader, optimizer, device, is_train, codebook_size, n_layers):
    model.train(is_train)
    totals = defaultdict(float)
    epoch_sids = [[] for _ in range(n_layers)]
    n_batches = 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        bar = tqdm(loader, leave=False, desc='train' if is_train else 'val')
        for rel_emb, emb, is_self in bar:
            emb, rel_emb, is_self = emb.to(device), rel_emb.to(device), is_self.to(device)
            out = model(emb, rel_emb, is_self)
            loss = out['loss']

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            totals['loss'] += loss.item()
            totals['r_loss'] += out['r_loss'].item()
            totals['q_loss'] += out['q_loss'].item()
            totals['con_loss'] += out['con_loss'].item() if isinstance(out['con_loss'], torch.Tensor) else 0.0
            n_batches += 1
            for lvl in range(n_layers):
                epoch_sids[lvl].append(out['sids'][lvl].detach().cpu())
            bar.set_postfix(loss=f'{loss.item():.4f}', rec=f'{out["r_loss"].item():.4f}')

    avg = {k: v / n_batches for k, v in totals.items()}
    avg['entropy'] = entropy_per_level(epoch_sids, codebook_size)
    avg['sids_num'] = torch.tensor([
        torch.cat(epoch_sids[l]).unique().numel() for l in range(n_layers)
    ])
    return avg


def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def compact_metric_dict(metrics):
    out = {}
    for k, v in metrics.items():
        if isinstance(v, torch.Tensor):
            if v.ndim == 0:
                out[k] = float(v.item())
            else:
                out[k] = [float(x) for x in v.detach().cpu().view(-1).tolist()]
        else:
            out[k] = float(v) if isinstance(v, (np.floating, np.integer)) else v
    return out


def history_to_cpu(history):
    clean = {}
    for k, values in history.items():
        clean[k] = []
        for v in values:
            if isinstance(v, torch.Tensor):
                clean[k].append(v.detach().cpu())
            else:
                clean[k].append(v)
    return clean


def hparams_dict(seed):
    return dict(
        seed=seed,
        n_layers=n_layers,
        codebook_size=codebook_size,
        embed_dim=embed_dim,
        hidden_sizes=hidden_sizes,
        beta=beta,
        gamma=gamma,
        ema_decay=ema_decay,
        temperature=temperature,
        lr=lr,
        n_ep=n_ep,
        patience=patience,
        min_delta=min_delta,
        batch_size=batch_size,
        checkpoint_epochs=checkpoint_epochs,
        main_tie_break=main_tie_break,
        gpt2_seeds=gpt2_seeds,
    )


def save_rqvae_checkpoint(path, model, optimizer, history, seed, epoch, tag,
                          train_metrics, val_metrics, best_val_loss, best_epoch):
    payload = {
        'model_state': cpu_state_dict(model),
        'optimizer_state': optimizer.state_dict(),
        'history': history_to_cpu(history),
        'seed': seed,
        'epoch': epoch,
        'tag': tag,
        'train_metrics': compact_metric_dict(train_metrics),
        'val_metrics': compact_metric_dict(val_metrics),
        'best_val_loss': float(best_val_loss),
        'best_epoch': int(best_epoch) if best_epoch is not None else None,
        'hparams': hparams_dict(seed),
    }
    torch.save(payload, path)
    return {
        'seed': seed,
        'checkpoint_tag': tag,
        'epoch': epoch,
        'checkpoint_path': str(path),
        'val_loss': float(val_metrics['loss']),
        'train_loss': float(train_metrics['loss']),
        'best_val_loss': float(best_val_loss),
        'best_epoch': int(best_epoch) if best_epoch is not None else None,
        'n_layers': n_layers,
        'codebook_size': codebook_size,
        'main_tie_break': main_tie_break,
        'gpt2_seeds': ','.join(map(str, gpt2_seeds)),
    }


In [ ]:
preview_model = make_model(seed=rqvae_seeds[0])
print(preview_model)
total_params = sum(p.numel() for p in preview_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
del preview_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def train_one_rqvae_seed(seed):
    set_global_seed(seed)
    model = make_model(seed)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    dl_train_seeded, dl_val_seeded = make_loaders(seed)

    history = defaultdict(list)
    records = []
    best_val_loss = float('inf')
    best_epoch = None
    best_state = None
    best_train_metrics = None
    best_val_metrics = None
    wait = 0
    last_epoch = 0
    last_train_metrics = None
    last_val_metrics = None

    for epoch in range(1, n_ep + 1):
        train_metrics = run_epoch(model, dl_train_seeded, optimizer, device, True, codebook_size, n_layers)
        val_metrics = run_epoch(model, dl_val_seeded, optimizer, device, False, codebook_size, n_layers)
        last_epoch = epoch
        last_train_metrics = train_metrics
        last_val_metrics = val_metrics

        for k, v in train_metrics.items():
            history[f'train_{k}'].append(v)
        for k, v in val_metrics.items():
            history[f'val_{k}'].append(v)

        improved = val_metrics['loss'] < best_val_loss - min_delta
        if improved:
            best_val_loss = val_metrics['loss']
            best_epoch = epoch
            best_state = cpu_state_dict(model)
            best_train_metrics = train_metrics
            best_val_metrics = val_metrics
            wait = 0

            best_path = RQVAE_CKPT_DIR / f'rqvae_l{n_layers}_seed{seed}_best.pt'
            torch.save({
                'model_state': best_state,
                'optimizer_state': optimizer.state_dict(),
                'history': history_to_cpu(history),
                'seed': seed,
                'epoch': epoch,
                'tag': 'best',
                'train_metrics': compact_metric_dict(train_metrics),
                'val_metrics': compact_metric_dict(val_metrics),
                'best_val_loss': float(best_val_loss),
                'best_epoch': int(best_epoch),
                'hparams': hparams_dict(seed),
            }, best_path)
        else:
            wait += 1

        if epoch in checkpoint_epochs:
            ckpt_path = RQVAE_CKPT_DIR / f'rqvae_l{n_layers}_seed{seed}_epoch{epoch:03d}.pt'
            records.append(save_rqvae_checkpoint(
                ckpt_path, model, optimizer, history, seed, epoch, f'epoch_{epoch:03d}',
                train_metrics, val_metrics, best_val_loss, best_epoch,
            ))

        if epoch in checkpoint_epochs or epoch % 20 == 0 or improved:
            print(
                f'seed={seed} epoch={epoch:03d}/{n_ep} '
                f'train={train_metrics["loss"]:.4f} val={val_metrics["loss"]:.4f} '
                f'best={best_val_loss:.4f} best_epoch={best_epoch} wait={wait}/{patience} '
                f'sids={val_metrics["sids_num"].tolist()} '
                f'entropy={[round(float(x), 2) for x in val_metrics["entropy"].tolist()]}'
            )

        if wait >= patience:
            print(f'Early stopping seed={seed} at epoch={epoch} best_epoch={best_epoch} best_val={best_val_loss:.6f}')
            break

    final_path = RQVAE_CKPT_DIR / f'rqvae_l{n_layers}_seed{seed}_final_epoch{last_epoch:03d}.pt'
    records.append(save_rqvae_checkpoint(
        final_path, model, optimizer, history, seed, last_epoch, 'final',
        last_train_metrics, last_val_metrics, best_val_loss, best_epoch,
    ))

    best_path = RQVAE_CKPT_DIR / f'rqvae_l{n_layers}_seed{seed}_best.pt'
    if best_state is not None:
        torch.save({
            'model_state': best_state,
            'optimizer_state': optimizer.state_dict(),
            'history': history_to_cpu(history),
            'seed': seed,
            'epoch': best_epoch,
            'tag': 'best',
            'train_metrics': compact_metric_dict(best_train_metrics),
            'val_metrics': compact_metric_dict(best_val_metrics),
            'best_val_loss': float(best_val_loss),
            'best_epoch': int(best_epoch),
            'hparams': hparams_dict(seed),
        }, best_path)
        records.append({
            'seed': seed,
            'checkpoint_tag': 'best',
            'epoch': int(best_epoch),
            'checkpoint_path': str(best_path),
            'val_loss': float(best_val_metrics['loss']),
            'train_loss': float(best_train_metrics['loss']),
            'best_val_loss': float(best_val_loss),
            'best_epoch': int(best_epoch),
            'n_layers': n_layers,
            'codebook_size': codebook_size,
            'main_tie_break': main_tie_break,
            'gpt2_seeds': ','.join(map(str, gpt2_seeds)),
        })
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return model, dict(history), records


all_manifest_records = []
all_history_by_seed = {}
last_model = None

for seed in rqvae_seeds:
    print(f'\n=== Training RQ-VAE seed {seed} ===')
    last_model, seed_history, seed_records = train_one_rqvae_seed(seed)
    all_history_by_seed[seed] = seed_history
    all_manifest_records.extend(seed_records)
    pd.DataFrame(all_manifest_records).to_csv(MANIFEST_PATH, index=False)
    print(f'Seed {seed} done. Manifest updated: {MANIFEST_PATH}')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

manifest_df = pd.DataFrame(all_manifest_records)
manifest_df.to_csv(MANIFEST_PATH, index=False)
display(manifest_df)


In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)

gpt2_plan_rows = []
for _, row in manifest_df.iterrows():
    for gpt2_seed in gpt2_seeds:
        gpt2_plan_rows.append({
            'rqvae_seed': int(row['seed']),
            'checkpoint_tag': row['checkpoint_tag'],
            'rqvae_epoch': int(row['epoch']),
            'rqvae_checkpoint_path': row['checkpoint_path'],
            'tie_break': main_tie_break,
            'gpt2_seed': gpt2_seed,
            'status': 'pending',
        })

gpt2_plan = pd.DataFrame(gpt2_plan_rows)
gpt2_plan.to_csv(GPT2_PLAN_PATH, index=False)
print('Saved GPT2Rec sweep plan:', GPT2_PLAN_PATH)
print('Expected GPT2Rec runs:', len(gpt2_plan))
display(gpt2_plan.head(20))

if last_model is not None:
    results = sids_analyze(last_model, ds, device)
    metrics = calculate_metrics(
        results['sid_to_ids'], len(ds),
        embeddings=embeds,
        codebook_size=codebook_size,
        n_layers=n_layers,
    )
    metrics['total_loss'] = results['total_loss']
    display(pd.DataFrame({'last_seed_best_model': metrics}).T)
else:
    print('No trained model in memory. Run the training cell first.')


In [ ]:
def plot_training(history, title='RQVAE Improved'):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(title, fontsize=14)

    axes[0, 0].plot(epochs, history['train_loss'], label='train')
    axes[0, 0].plot(epochs, history['val_loss'], label='val')
    axes[0, 0].set_yscale('log')
    axes[0, 0].set_title('Total loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()

    axes[0, 1].plot(epochs, history['train_r_loss'], label='recon')
    axes[0, 1].plot(epochs, history['train_q_loss'], label='quant')
    axes[0, 1].plot(epochs, history['train_con_loss'], label='contrastive')
    axes[0, 1].set_yscale('log')
    axes[0, 1].set_title('Train loss components')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()

    sids_tr = np.array([t.numpy() for t in history['train_sids_num']])
    sids_va = np.array([t.numpy() for t in history['val_sids_num']])
    for lvl in range(sids_tr.shape[1]):
        axes[1, 0].plot(epochs, sids_tr[:, lvl], label=f'train L{lvl}')
        axes[1, 0].plot(epochs, sids_va[:, lvl], '--', label=f'val L{lvl}')
    axes[1, 0].set_title('Unique SIDs per level')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend(fontsize=7)

    ent_tr = np.array([t.numpy() for t in history['train_entropy']])
    ent_va = np.array([t.numpy() for t in history['val_entropy']])
    for lvl in range(ent_tr.shape[1]):
        axes[1, 1].plot(epochs, ent_tr[:, lvl], label=f'train L{lvl}')
        axes[1, 1].plot(epochs, ent_va[:, lvl], '--', label=f'val L{lvl}')
    axes[1, 1].axhline(np.log2(codebook_size), color='k', ls=':', label=f'max log2({codebook_size})')
    axes[1, 1].set_title('Codebook entropy per level')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend(fontsize=7)

    plt.tight_layout()
    out_path = RQVAE_CKPT_DIR / f'{title.replace(" ", "_").lower()}_training.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print('Saved plot:', out_path)
    plt.show()

if 'all_history_by_seed' in globals() and all_history_by_seed:
    for seed, history in all_history_by_seed.items():
        plot_training(history, title=f'RQVAE seed {seed}')
else:
    print('No training history in memory. Run the training cell first.')
